# Practice: Build a Classifier You Could Defend

**Computational Text Analysis Pack · Module 07 — Classification**

*Technique:* A supervised study end to end: baseline, model, errors, and honest reporting

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/06-supervised-classification/07d_practice_classifier.ipynb)

Data: `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## The question

> **Could you use a classifier to sort 50,000 unlabelled African news articles
> into topics for a research corpus — and what would you have to report for a
> reader to trust the result?**

Building the model is the easy part. This notebook spends most of its time on
the part that makes it usable as evidence.

### How to use this notebook

A worked case, not a tutorial. Run it once for the shape of the argument, then
change the question at the top to one of your own.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. It does not
    match combining marks, and a tone mark on ọ or ẹ is always one, so the
    pattern adds the combining-accent range U+0300-U+036F; without it ọ̀rọ̀
    splits into ọ + rọ. The common shortcut `[a-z]+` throws away far more and
    quietly changes every count in this notebook — see notebook 03b.
    """
    return re.findall(r"[\w\u0300-\u036f]+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé lórí ọ̀rọ̀ ẹ̀kọ́ — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("bare \\w+   :", re.findall(r"\w+", sample.lower()))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter; bare \\w+")
print("splits them wherever a tone mark sits on ọ or ẹ.")

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix)
from sklearn.pipeline import make_pipeline

sub = df[df["n_words"] >= 30].reset_index(drop=True)
Xtr, Xte, ytr, yte = train_test_split(sub["text"], sub["category"], test_size=0.25,
                                      random_state=0, stratify=sub["category"])

# Step 1: the baseline you must beat. Always report it.
majority = yte.value_counts(normalize=True).max()
print(f"majority-class baseline: {majority:.3f}")
print(f"class balance:\n{sub['category'].value_counts(normalize=True).round(3).to_string()}")

In [ ]:
# Step 2: the model, inside a pipeline so the vectoriser never sees test data.
pipe = make_pipeline(
    TfidfVectorizer(token_pattern=r"[\w\u0300-\u036f]+", min_df=3, max_features=30000, sublinear_tf=True),
    LogisticRegression(max_iter=1000, class_weight="balanced"))
pipe.fit(Xtr, ytr)
pred = pipe.predict(Xte)
print(f"held-out accuracy : {accuracy_score(yte, pred):.3f}")
print(f"macro F1          : {f1_score(yte, pred, average='macro'):.3f}")
print()
cv = cross_val_score(pipe, sub["text"], sub["category"],
                     cv=StratifiedKFold(4, shuffle=True, random_state=0), n_jobs=1)
print(f"4-fold CV         : {cv.mean():.3f} +/- {cv.std():.3f}")
print()
print("Report the spread, not just the mean. A model whose folds disagree by")
print("several points is not a model you can quote to three decimal places.")

In [ ]:
# Step 3: per class, because macro F1 hides the classes that fail.
print(classification_report(yte, pred, zero_division=0))
print("Look at the support column. The classes the model does worst on are")
print("usually the rare ones — and in a research corpus those may be exactly")
print("the ones you care about.")

In [ ]:
# Step 4: read the errors. This is the step that gets skipped.
cats = sorted(sub["category"].unique())
cm = confusion_matrix(yte, pred, labels=cats)
print(pd.DataFrame(cm, index=cats, columns=cats).to_string())
print()
off = [(cats[i], cats[j], cm[i][j]) for i in range(len(cats)) for j in range(len(cats)) if i != j]
off.sort(key=lambda t: -t[2])
print("largest confusions:")
for a, b, n in off[:4]:
    print(f"   {a} -> {b}: {n}")

worst_true, worst_pred, _ = off[0]
mask = (yte.values == worst_true) & (pred == worst_pred)
examples = Xte.values[mask][:2]
print(f"\nexample documents labelled '{worst_true}' but predicted '{worst_pred}':\n")
for t in examples:
    print("   " + " ".join(str(t).split())[:220] + "...\n")
print("Read them. Often the 'error' is a document that genuinely belongs to")
print("both categories, which is a fact about your label scheme, not the model.")

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5))
im = ax.imshow(cm / cm.sum(axis=1, keepdims=True), cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(cats))); ax.set_xticklabels(cats, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(cats))); ax.set_yticklabels(cats, fontsize=8)
ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title("Row-normalised confusion")
fig.colorbar(im, ax=ax, shrink=.8); plt.tight_layout()

## What you would have to report

To use this on 50,000 unlabelled documents and have a reader trust it:

1. **The baseline** (majority class), so the accuracy has a floor to be read against.
2. **Cross-validation with its spread**, not a single split.
3. **Per-class scores**, because the aggregate hides the weak classes.
4. **Per-language scores** if the corpus is multilingual — module 26b's lesson.
5. **The confusion structure**, and whether the top confusion is a model failure
   or a label-scheme problem.
6. **What you did with the unlabelled predictions.** Filtering a corpus by a
   model at 0.8 accuracy means roughly one document in five is misfiled, and any
   downstream count inherits that.

**Not defensible:** "we classified the corpus with 85% accuracy" as a standalone
sentence. It omits every number above.

### 🔧 Your turn 1

Set `class_weight=None` and re-run. Which classes improve and which collapse?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

Accuracy usually rises slightly while macro F1 falls, because the model buys the
gain by predicting the common classes more often and abandoning the rare ones.
Whether that is an improvement depends entirely on your research question: for
building a corpus of the rare category it is a disaster, and for estimating the
overall distribution it may be fine. This is why the metric has to be chosen
from the question rather than from convention.

</details>